# Create Study Dataset

Here, we use th 2015-2019 data extracted using the Chakraborty et al. method. We aggregate this biophysical data with demographic data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
uhi = pd.read_csv('data/uhi_chakraborty.csv')
cej = pd.read_csv('data/CEJST-v1.csv', low_memory=False)

In [3]:
cej = cej[['Census tract 2010 ID', 'Percent Black or African American alone',
           'Percent Hispanic or Latino', 'Percent White',
           'Is low income?', 'Percent of individuals < 100% Federal Poverty Line']]

cols = {'Percent Black or African American alone': 'Black',  
        'Percent Hispanic or Latino': 'Hispanic', 'Percent White': 'White',
        'Percent of individuals < 100% Federal Poverty Line': 'Below Poverty'}

cej.rename(columns=cols, inplace=True)

In [4]:
len(uhi)

55871

In [5]:
uhi['SUHI'] = uhi['Summer_LST_Day'] - uhi['Summer_LST_Day_rur']
uhi['DEM'] = uhi['Ele'] - uhi['Ele_rur']
uhi['NDVI'] = uhi['Summer_NDVI'] - uhi['Summer_NDVI_rur']
uhi['BSA'] = uhi['Summer_BSA'] - uhi['Summer_BSA_rur']
uhi['NDBI'] = uhi['Summer_NDBI'] - uhi['Summer_NDBI_rur']

In [6]:
uhi.head()

,COUNTY,Clim_zone,Coastal?,FIPS,STATE,Summer_NDBI,Summer_NDVI,Summer_BSA,Ele,Summer_LST_Day,Census_geoid,Urban_geoid,Urban_name,Urban_name_extended,Ele_rur,Summer_BSA_rur,Summer_LST_Day_rur,Summer_NDBI_rur,Summer_NDVI_rur,SUHI,DEM,NDVI,BSA,NDBI
0,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.100101,0.430313,0.156244,529.969630,42.927103,48441011400,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,1.886648,-57.477455,0.012692,0.001273,-0.127350
1,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.039614,0.412304,0.173403,530.453409,41.154747,48441012200,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,0.114292,-56.993676,-0.005317,0.018432,-0.066863
2,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.039671,0.430774,0.159267,528.556320,39.993913,48441012100,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,-1.046542,-58.890764,0.013153,0.004297,-0.066920
3,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.029324,0.359078,0.170724,528.426211,43.409921,48441011900,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,2.369466,-59.020873,-0.058543,0.015753,-0.056574
4,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.049882,0.398955,0.160218,536.292313,42.957912,48441012300,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,1.917457,-51.154771,-0.018666,0.005248,-0.077131


In [7]:
data = uhi.merge(cej, left_on='Census_geoid', right_on='Census tract 2010 ID', how='left')

In [8]:
data.rename(columns={'Clim_zone': 'Climate Zone'}, inplace=True)

In [9]:
len(data)

55871

In [10]:
data['Black']    = data['Black'] * 100
data['Hispanic'] = data['Hispanic'] * 100
data['White']    = data['White'] * 100

## Add Median Income Data

In [11]:
income_df = pd.read_csv('data/ACSDT5Y2019.B20018-Data.csv', skiprows=1)
income_df['Census_geoid'] = income_df['Geography'].str[9:]
income_df['Census_geoid'] = income_df['Census_geoid'].astype('int64')
income_df.rename(columns={'Estimate!!Median earnings in the past 12 months (in 2019 inflation-adjusted dollars)': 'Median Income'}, inplace=True)

data = data.merge(income_df[['Census_geoid', 'Median Income']], how='left', on='Census_geoid')

In [12]:
data.loc[data['Median Income'] == '-', 'Median Income'] = np.nan

data['Median Income'] = data['Median Income'].str.replace('+', '', regex=True)
data['Median Income'] = data['Median Income'].str.replace('-', '', regex=True)
data['Median Income'] = data['Median Income'].str.replace(',', '', regex=True)

# Fill null values with a temporary placeholder
data['Median Income'] = data['Median Income'].fillna(-999)

# Convert to integer
data['Median Income'] = data['Median Income'].astype('int64')

# Set the placeholder back to nulls
data['Median Income'] = data['Median Income'].replace(-999, np.nan)

## Normalize demographic data by city as percentiles

In [13]:
data['Median Income Percentile'] = data.groupby('Urban_name')['Median Income'].rank(pct=True)
data['Black Percentile'] = data.groupby('Urban_name')['Black'].rank(pct=True)
data['Hispanic Percentile'] = data.groupby('Urban_name')['Hispanic'].rank(pct=True)
data['White Percentile'] = data.groupby('Urban_name')['White'].rank(pct=True)


In [14]:
data.head()

,COUNTY,Climate Zone,Coastal?,FIPS,STATE,Summer_NDBI,Summer_NDVI,Summer_BSA,Ele,Summer_LST_Day,Census_geoid,Urban_geoid,Urban_name,Urban_name_extended,Ele_rur,Summer_BSA_rur,Summer_LST_Day_rur,Summer_NDBI_rur,Summer_NDVI_rur,SUHI,DEM,NDVI,BSA,NDBI,Census tract 2010 ID,Black,Hispanic,White,Is low income?,Below Poverty,Median Income,Median Income Percentile,Black Percentile,Hispanic Percentile,White Percentile
0,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.100101,0.430313,0.156244,529.969630,42.927103,48441011400,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,1.886648,-57.477455,0.012692,0.001273,-0.127350,48441011400,11.0,20.0,62.0,True,17.0,39444.0,0.742857,0.708333,0.444444,0.583333
1,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.039614,0.412304,0.173403,530.453409,41.154747,48441012200,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,0.114292,-56.993676,-0.005317,0.018432,-0.066863,48441012200,8.0,33.0,56.0,True,9.0,33150.0,0.428571,0.513889,0.694444,0.472222
2,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.039671,0.430774,0.159267,528.556320,39.993913,48441012100,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,-1.046542,-58.890764,0.013153,0.004297,-0.066920,48441012100,12.0,8.0,76.0,True,54.0,NaN,NaN,0.750000,0.069444,0.888889
3,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.029324,0.359078,0.170724,528.426211,43.409921,48441011900,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,2.369466,-59.020873,-0.058543,0.015753,-0.056574,48441011900,4.0,56.0,37.0,True,33.0,32294.0,0.371429,0.263889,0.916667,0.138889
4,Taylor,Temperate,False,4.844101e+10,TEXAS,-0.049882,0.398955,0.160218,536.292313,42.957912,48441012300,280,"Abilene, TX","Abilene, TX Urbanized Area",587.447084,0.154971,41.040455,0.027249,0.417621,1.917457,-51.154771,-0.018666,0.005248,-0.077131,48441012300,11.0,26.0,55.0,False,24.0,32957.0,0.400000,0.708333,0.583333,0.444444


In [15]:
d_stats = data[['BSA', 'NDBI', 'NDVI', 'DEM', 'SUHI']].describe().round(4)
d_stats.T

,count,mean,std,min,25%,50%,75%,max
BSA,51854.0,-0.0059,0.0170,-0.1305,-0.0153,-0.0055,0.0039,0.0881
NDBI,55845.0,0.0407,0.0887,-0.3018,-0.0114,0.0336,0.0878,0.6889
NDVI,55845.0,-0.1361,0.1353,-0.7292,-0.2032,-0.1129,-0.0379,0.3343
DEM,55846.0,-107.1220,211.0684,-1021.9568,-96.0924,-29.8976,-1.0296,1038.9740
SUHI,55638.0,3.5436,3.1250,-12.4765,1.2125,3.5739,5.8142,14.3835


In [16]:
missing = pd.DataFrame({'Total Missing': data.isna().sum(), 'Percent Missing': (data.isna().sum() / len(data)) * 100 })
missing

,Total Missing,Percent Missing
COUNTY,25,0.044746
Climate Zone,25,0.044746
Coastal?,25,0.044746
FIPS,25,0.044746
STATE,25,0.044746
Summer_NDBI,26,0.046536
Summer_NDVI,26,0.046536
Summer_BSA,4017,7.189776
Ele,25,0.044746
Summer_LST_Day,233,0.417032


In [17]:
data.to_csv('data/data_sue.csv', index=False)